# Program *Training*: Jaringan Saraf Tiruan dengan *Back Propagation*

Mengenali 3 kelas kendaraan (pesawat, mobil, kapal) dari citra *grayscale* 32 x 32.
Algoritma dan notasi mengikuti Fausett, *Fundamentals of Neural Networks* (1994), Bab 6, Langkah 0 sampai 9.

Hasil akhir: satu berkas `BOBOT.npz` berisi `v`, `v_0`, `w`, `w_0`.

## Sel 1. Persiapan dan pengaturan

Menyambungkan Google Drive dan menetapkan semua parameter di satu tempat. Ubah nilainya di sini bila perlu.

In [ ]:
import os
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

# Lokasi di Google Drive
FOLDER_PROYEK = '/content/drive/MyDrive/TUGAS_ALGORITMA_AI'
FOLDER_LATIH  = FOLDER_PROYEK + '/DATASET/LATIH'
FOLDER_UJI    = FOLDER_PROYEK + '/DATASET/UJI'
BERKAS_BOBOT  = FOLDER_PROYEK + '/BOBOT/BOBOT.npz'

# Ukuran gambar dan arsitektur
LEBAR, TINGGI = 32, 32
n = LEBAR * TINGGI      # jumlah node input
p = 100                 # jumlah node hidden
m = 3                   # jumlah node output (harus tepat 3)

# Parameter pembelajaran
alpha       = 0.1       # laju pembelajaran
EPOCH_MAKS  = 200       # batas epoch (syarat berhenti 1)
AMBANG_GALAT = 0.01     # galat rata-rata per pola (syarat berhenti 2)
BATAS_ACAK  = 0.5       # bobot awal acak di antara -0,5 dan 0,5
BIJI_ACAK   = 42
CETAK_TIAP  = 10        # cetak laporan tiap sekian epoch

NAMA_KELAS = ['KELAS1_PESAWAT', 'KELAS2_MOBIL', 'KELAS3_KAPAL']

print('=' * 52)
print('PENGATURAN')
print('=' * 52)
print(f'Node input  (n) : {n}  ({LEBAR} x {TINGGI} pixel)')
print(f'Node hidden (p) : {p}')
print(f'Node output (m) : {m}')
print(f'alpha           : {alpha}')
print(f'Epoch maksimum  : {EPOCH_MAKS}')
print(f'Ambang galat    : {AMBANG_GALAT}')

## Sel 2. *Preprocessing* dan memuat *dataset*

Setiap gambar RGB diubah menjadi *grayscale*, diratakan menjadi vektor berukuran 1024, lalu dinormalisasi ke rentang 0 sampai 1.
Target `t` ditentukan dari nama folder kelas: `1 0 0`, `0 1 0`, `0 0 1`.

In [ ]:
def praproses(gambar):
    """Gambar PIL (RGB) -> vektor x berukuran n, nilai 0 sampai 1."""
    abu = gambar.convert('L')                       # 1. grayscale
    if abu.size != (LEBAR, TINGGI):                 # 2. pengaman ukuran
        abu = abu.resize((LEBAR, TINGGI))
    x = np.asarray(abu, dtype=np.float64).flatten() # 3. ratakan
    return x / 255.0                                # 4. normalisasi


def muat_dataset(folder):
    X, T, label, nama_berkas = [], [], [], []
    for k, nama in enumerate(NAMA_KELAS):
        daftar = sorted(f for f in os.listdir(f'{folder}/{nama}') if f.endswith('.png'))
        for berkas in daftar:
            gambar = Image.open(f'{folder}/{nama}/{berkas}')
            X.append(praproses(gambar))
            target = np.zeros(m)
            target[k] = 1.0                         # 100, 010, atau 001
            T.append(target)
            label.append(k)
            nama_berkas.append(f'{nama}/{berkas}')
    return np.array(X), np.array(T), np.array(label), nama_berkas


X_latih, T_latih, label_latih, _ = muat_dataset(FOLDER_LATIH)
X_uji,   T_uji,   label_uji,   _ = muat_dataset(FOLDER_UJI)

print('=' * 52)
print('DATASET DIMUAT')
print('=' * 52)
print(f'{"Kelas":<18}{"Latih":>8}{"Uji":>8}   Target')
for k, nama in enumerate(NAMA_KELAS):
    target = ' '.join(str(int(a)) for a in np.eye(m)[k])
    print(f'{nama:<18}{(label_latih == k).sum():>8}{(label_uji == k).sum():>8}   {target}')
print('-' * 52)
print(f'{"Total":<18}{len(X_latih):>8}{len(X_uji):>8}')
print(f'\nBentuk X_latih : {X_latih.shape}  (pola x node input)')
print(f'Rentang nilai  : {X_latih.min():.2f} sampai {X_latih.max():.2f}')

## Sel 3. Bukti konversi *grayscale*

Baris atas gambar asli (RGB), baris bawah hasil *grayscale* yang masuk ke jaringan.

In [ ]:
fig, sumbu = plt.subplots(2, 3, figsize=(7, 5))
for k, nama in enumerate(NAMA_KELAS):
    berkas = sorted(f for f in os.listdir(f'{FOLDER_LATIH}/{nama}') if f.endswith('.png'))[0]
    asli = Image.open(f'{FOLDER_LATIH}/{nama}/{berkas}')
    sumbu[0, k].imshow(asli)
    sumbu[0, k].set_title(nama.split('_')[1] + f'\n(mode {asli.mode})')
    sumbu[1, k].imshow(praproses(asli).reshape(TINGGI, LEBAR), cmap='gray', vmin=0, vmax=1)
    sumbu[1, k].set_title('grayscale')
    sumbu[0, k].axis('off'); sumbu[1, k].axis('off')
plt.tight_layout()
plt.show()

## Sel 4. Fungsi aktivasi dan inisialisasi bobot (Langkah 0)

Sigmoid biner `f(x) = 1 / (1 + e^-x)` dan turunannya `f'(x) = f(x)(1 - f(x))`.
Semua bobot dan bias diisi bilangan acak kecil.

In [ ]:
def f(x):
    return 1.0 / (1.0 + np.exp(-x))

def f_aksen(x):
    return f(x) * (1.0 - f(x))


# Langkah 0: inisialisasi bobot
rng = np.random.default_rng(BIJI_ACAK)
v   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (n, p))   # input  -> hidden
v_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, p)        # bias hidden
w   = rng.uniform(-BATAS_ACAK, BATAS_ACAK, (p, m))   # hidden -> output
w_0 = rng.uniform(-BATAS_ACAK, BATAS_ACAK, m)        # bias output

print('=' * 52)
print('BOBOT AWAL (Langkah 0)')
print('=' * 52)
print(f'{"Variabel":<10}{"Ukuran":<14}{"Jumlah nilai":>12}')
for nama, larik in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)]:
    print(f'{nama:<10}{str(larik.shape):<14}{larik.size:>12,}')
print(f'\nRentang bobot awal: {min(v.min(), w.min()):.2f} sampai {max(v.max(), w.max()):.2f}')

## Sel 5. *Training* (Langkah 1 sampai 9)

Tiap pola dikerjakan dengan Langkah 3 sampai 8, bobot diperbarui per pola (sesuai buku).
Urutan pola diacak setiap *epoch*. Berhenti bila *epoch* maksimum tercapai atau galat rata-rata di bawah ambang.

In [ ]:
def umpan_maju(X):
    """Hanya untuk melaporkan kemajuan (tidak mengubah bobot)."""
    z = f(v_0 + X @ v)
    return f(w_0 + z @ w)

def akurasi(X, label):
    return np.mean(np.argmax(umpan_maju(X), axis=1) == label) * 100


riwayat_galat, riwayat_akurasi = [], []
jumlah_pola = len(X_latih)

print('=' * 52)
print('TRAINING')
print('=' * 52)
print(f'{"Epoch":>6}{"Galat rata-rata":>18}{"Akurasi latih":>16}')
print('-' * 52)

waktu_mulai = time.time()

# Langkah 1: ulangi selama syarat berhenti belum terpenuhi
for epoch in range(1, EPOCH_MAKS + 1):
    galat_total = 0.0

    # Langkah 2: untuk setiap pasangan (x, t)
    for s in rng.permutation(jumlah_pola):
        x = X_latih[s]
        t = T_latih[s]

        # Langkah 3 sampai 5: umpan maju
        z_in = v_0 + x @ v
        z    = f(z_in)
        y_in = w_0 + z @ w
        y    = f(y_in)

        # Langkah 6: galat di node output
        delta_k   = (t - y) * f_aksen(y_in)
        delta_w   = alpha * np.outer(z, delta_k)
        delta_w_0 = alpha * delta_k

        # Langkah 7: galat di node hidden (memakai w lama)
        delta_in_j = w @ delta_k
        delta_j    = delta_in_j * f_aksen(z_in)
        delta_v    = alpha * np.outer(x, delta_j)
        delta_v_0  = alpha * delta_j

        # Langkah 8: perbarui bobot dan bias
        w   = w   + delta_w
        w_0 = w_0 + delta_w_0
        v   = v   + delta_v
        v_0 = v_0 + delta_v_0

        galat_total += 0.5 * np.sum((t - y) ** 2)

    galat_rata = galat_total / jumlah_pola
    riwayat_galat.append(galat_rata)
    riwayat_akurasi.append(akurasi(X_latih, label_latih))

    if epoch == 1:
        print(f'(waktu 1 epoch: {time.time() - waktu_mulai:.1f} detik)')
    if epoch == 1 or epoch % CETAK_TIAP == 0:
        print(f'{epoch:>6}{galat_rata:>18.5f}{riwayat_akurasi[-1]:>15.1f}%')

    # Langkah 9: uji syarat berhenti
    if galat_rata < AMBANG_GALAT:
        print(f'\nBerhenti: galat rata-rata di bawah ambang pada epoch {epoch}.')
        break
else:
    print(f'\nBerhenti: epoch maksimum ({EPOCH_MAKS}) tercapai.')

print('-' * 52)
print(f'Epoch dijalankan : {len(riwayat_galat)}')
print(f'Galat akhir      : {riwayat_galat[-1]:.5f}')
print(f'Akurasi latih    : {riwayat_akurasi[-1]:.1f}%')
print(f'Waktu total      : {time.time() - waktu_mulai:.1f} detik')

## Sel 6. Grafik kemajuan *training*

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
a1.plot(range(1, len(riwayat_galat) + 1), riwayat_galat)
a1.set_title('Galat rata-rata per epoch'); a1.set_xlabel('Epoch'); a1.set_ylabel('Galat'); a1.grid(alpha=0.3)
a2.plot(range(1, len(riwayat_akurasi) + 1), riwayat_akurasi, color='tab:green')
a2.set_title('Akurasi data latih (%)'); a2.set_xlabel('Epoch'); a2.set_ylim(0, 100); a2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Sel 7. Evaluasi pada data latih dan data uji

Data uji tidak ikut dilatih. Selisih besar antara akurasi latih dan uji menandakan *overfitting*.
Baris pada tabel adalah kelas sebenarnya, kolom adalah tebakan jaringan.

In [ ]:
def laporan(X, label, judul):
    tebakan = np.argmax(umpan_maju(X), axis=1)
    benar = np.mean(tebakan == label) * 100
    print(f'{judul}: {benar:.1f}%  ({int((tebakan == label).sum())} dari {len(label)} gambar benar)')
    return tebakan

print('=' * 52)
print('EVALUASI')
print('=' * 52)
laporan(X_latih, label_latih, 'Akurasi data latih')
tebakan_uji = laporan(X_uji, label_uji, 'Akurasi data uji  ')

print('\nTabel tebakan pada data uji')
judul = [nama.split('_')[1] for nama in NAMA_KELAS]
print(f'{"Sebenarnya":<12}' + ''.join(f'{j:>9}' for j in judul) + f'{"Akurasi":>10}')
for k in range(m):
    baris = [int(np.sum((label_uji == k) & (tebakan_uji == j))) for j in range(m)]
    persen = baris[k] / np.sum(label_uji == k) * 100
    print(f'{judul[k]:<12}' + ''.join(f'{b:>9}' for b in baris) + f'{persen:>9.0f}%')

## Sel 8. Pola target dan keluaran jaringan

Bagian atas menunjukkan pola target `t` untuk tiap kelas. Bagian bawah membandingkan target dengan keluaran jaringan `y` pada beberapa gambar uji.
Keluaran yang baik: angka pada posisi yang seharusnya hampir 1, dua angka lainnya hampir 0.

In [ ]:
print('=' * 62)
print('POLA TARGET OUTPUT (t)')
print('=' * 62)
print(f'{"Kelas":<18}{"Y1":>4}{"Y2":>4}{"Y3":>4}   Artinya')
for k in range(m):
    pola = ''.join(f'{int(a):>4}' for a in np.eye(m)[k])
    print(f'{NAMA_KELAS[k]:<18}{pola}   hanya Y{k + 1} bernilai 1')

print()
print('=' * 62)
print('KELUARAN JARINGAN (y) DIBANDINGKAN DENGAN TARGET (t), DATA UJI')
print('=' * 62)
print(f'{"Kelas":<16}{"Target t":<10}{"Keluaran y (Y1, Y2, Y3)":<26}{"Hasil"}')
print('-' * 62)
y_uji = umpan_maju(X_uji)
for k in range(m):
    for s in np.where(label_uji == k)[0][:3]:
        t_teks = ' '.join(str(int(a)) for a in T_uji[s])
        y_teks = '  '.join(f'{a:.3f}' for a in y_uji[s])
        hasil = 'BENAR' if np.argmax(y_uji[s]) == k else 'SALAH'
        print(f'{NAMA_KELAS[k].split('_')[1]:<16}{t_teks:<10}{y_teks:<26}{hasil}')

## Sel 9. Menyimpan bobot

Bobot `v`, `v_0`, `w`, `w_0` disimpan dalam satu berkas `BOBOT.npz` di folder `BOBOT` pada Google Drive.
Program pengujian akan membaca berkas ini.

In [ ]:
os.makedirs(os.path.dirname(BERKAS_BOBOT), exist_ok=True)
np.savez(BERKAS_BOBOT, v=v, v_0=v_0, w=w, w_0=w_0, ukuran=np.array([LEBAR, TINGGI]))

# Pemeriksaan: baca ulang berkas yang baru disimpan
cek = np.load(BERKAS_BOBOT)
print('=' * 52)
print('BOBOT TERSIMPAN')
print('=' * 52)
print(f'Lokasi : {BERKAS_BOBOT}')
print(f'Ukuran : {os.path.getsize(BERKAS_BOBOT) / 1024:.0f} KB\n')
print(f'{"Isi":<10}{"Ukuran":<14}')
for kunci in ['v', 'v_0', 'w', 'w_0']:
    print(f'{kunci:<10}{str(cek[kunci].shape):<14}')
sama = all(np.array_equal(cek[k], a) for k, a in [('v', v), ('v_0', v_0), ('w', w), ('w_0', w_0)])
print(f'\nIsi berkas sama dengan bobot di memori: {"YA" if sama else "TIDAK"}')